In [ ]:
import os

In [49]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.messages import ToolMessage
from langchain_core.rate_limiters import InMemoryRateLimiter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
from langchain.agents.middleware import AgentMiddleware, ModelCallLimitMiddleware, PIIMiddleware, TodoListMiddleware

In [25]:
class TolerateToolErrors(AgentMiddleware):
    """Hand tool failures back to the model as a message so it can recover, rather than
    crashing the run. Tools that touch the outside world, like a browser, fail now and then."""

    async def awrap_tool_call(self, request, handler):
        try:
            return await handler(request)
        except Exception as error:
            return ToolMessage(
                content=f"That tool call failed: {error}. Try another approach.",
                tool_call_id=request.tool_call["id"],
            )

In [4]:
client = MultiServerMCPClient(
    {
        "playwright": {
            "transport": "stdio",
            "command": "npx",
            "args": ["@playwright/mcp@latest", "--isolated"]
        }
    }
)

In [22]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


In [12]:
load_dotenv(override=True)

True

In [13]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

In [ ]:
class Essentials(BaseModel):
    smarket: str = Field(description="The Supermarkets name")
    region: str = Field(description="The region in which the prices are pulled from")
    prices: dict[str, float] = Field(description="The prices of essential items (cheapest): milk (2 litres), cheese (1kg), mince (500g), eggs (6 pack)")
    total_price: float = Field(description="The total cost of the essentials")

In [ ]:
mcp_tools = await client.get_tools()
tools = [search] + mcp_tools

In [55]:
SYSTEM_PROMPT = """You are a helpful supermarket assistant tasked with finding prices from the web.
Use the region given to you to find the prices or default to Auckland CBD. Use non member prices.
You explore different supermarket websites in New Zealand finding the prices for these essential items:
standard blue top milk(2 litres), cheese(1kg), beef mince(500g) 5% fat, eggs(6 pack). 
Get the cheapest branded item but make sure it conforms to the specified weight/amount shown in brackets.
After all prices found add them together for the total_price"""

In [45]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.5,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [26]:
memory = InMemorySaver()

In [50]:
model = ChatOpenAI(
    model="gpt-5.4-mini",
    rate_limiter=rate_limiter
)

In [51]:
woolies_agents = create_agent(
    model=model, 
    tools=tools, 
    system_prompt=SYSTEM_PROMPT, 
    middleware=[TolerateToolErrors(), TodoListMiddleware(), PIIMiddleware("email"), PIIMiddleware("credit_card", apply_to_tool_results=True), ModelCallLimitMiddleware(run_limit=30)],
    checkpointer=memory,
    )

In [52]:
config = {"configurable": {"thread_id": "big3"}}
result = await woolies_agents.ainvoke({"messages": [{"role": "user", "content": "Get the essentials from woolworths NZ"}]}, config)

In [53]:
print(result["messages"][-1].content)

Here are the essentials from Woolworths NZ:

- Blue top milk, 2L: Woolworths Milk Standard Blue Top 2L — NZ$4.81
- Cheese, 1kg: Woolworths Cheese Cheddar Everyday Block 1kg — NZ$12.49
- Beef mince, 500g: Woolworths NZ Beef Mince Grass Fed 5% Fat Tray 500g — NZ$16.99
- Eggs, 6 pack: Woolworths Free Range Eggs Half Dozen Size 6 — NZ$6.20

Total price: NZ$40.49


